In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Imports
import mlflow
import mlflow.sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib

In [12]:
#leer archivo limpio 
df = pd.read_csv('airbnb_rio_clean.csv')

In [2]:
#explorar df
df.head()


,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_identity_verified,latitude,longitude,room_type,accommodates,bedrooms,...,has_pool,has_wifi,has_free_parking,has_ac_heating,has_kitchen,has_jacuzzi,has_washer_dryer,has_self_checkin,has_tv_cable,has_bbq
0,95,97,0.0,10.0,1.0,-22.982818,-43.222457,1.0,4,2.0,...,0,1,1,1,1,0,1,0,1,0
1,100,46,0.0,39.0,1.0,-22.984090,-43.191770,1.0,2,1.0,...,0,1,0,1,1,0,1,0,0,0
2,0,0,0.0,2.0,1.0,-22.814911,-43.379011,1.0,4,1.0,...,0,0,1,1,1,0,1,0,0,1
3,100,71,1.0,1.0,1.0,-23.010000,-43.344820,1.0,4,1.0,...,1,1,1,1,1,0,1,0,0,0
4,100,99,1.0,17.0,1.0,-22.970696,-43.186048,1.0,6,2.0,...,0,1,0,1,1,0,1,0,0,0


In [7]:
df.dtypes

host_response_rate               int64
host_acceptance_rate             int64
host_is_superhost              float64
host_total_listings_count      float64
host_identity_verified         float64
latitude                       float64
longitude                      float64
room_type                      float64
accommodates                     int64
bedrooms                       float64
beds                           float64
price                          float64
minimum_nights                   int64
maximum_nights                   int64
maximum_minimum_nights         float64
maximum_nights_avg_ntm         float64
availability_365                 int64
availability_eoy                 int64
estimated_occupancy_l365d        int64
review_scores_rating           float64
review_scores_communication    float64
review_scores_location         float64
instant_bookable                 int64
bathroomsf                     float64
has_pool                         int64
has_wifi                 

In [ ]:
#crear variable artificial
df["recomendable"] = (df["review_scores_rating"] >= 4.6).astype(int) #Definir umbral de clasificacion
df["recomendable"].value_counts(normalize=True) * 100 # ver porcentaje de datos 


In [11]:
bins = [0, 1, 2, 3, 4, 5]
labels = ["[0-1)", "[1-2)", "[2-3)", "[3-4)", "[4-5)"]

df["rating_range"] = pd.cut(df["review_scores_rating"], bins=bins, labels=labels, right=False)

df["rating_range"].value_counts(normalize=True).mul(100).round(2)

rating_range
[4-5)    68.62
[0-1)    29.45
[3-4)     1.37
[2-3)     0.29
[1-2)     0.28
Name: proportion, dtype: float64

## MODELO CLASIFICACION

In [ ]:
#Escoger variables que entran

x_1=[]

In [ ]:
'0. Funcion para estandarizar los datos numericos solo'
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

preprocesamiento = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), x_num),   # escalo solo las continuas--------------->
        # las categóricas (0/1) pasan sin tocarse
    ],remainder="passthrough")

'1. Dividir test y train'
#  Definir X e y
y = df["recomendable"].copy()
X = df[x_1].copy() # CAMBIARRRRRRR---------------------------->

# 4. Dividir en train y test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True )

'2. Estandarizar x num (se asume que categoricos ya estan en dummies)'
# Ajustar el preprocesador con datos de entrenamiento
preprocesamiento.fit(X_train)

# Transformar train y test --> estandariza
X_train_proc = preprocesamiento.transform(X_train)
X_test_proc = preprocesamiento.transform(X_test)

In [ ]:

# MLflow
mlflow.set_registry_uri("databricks-uc")
experiment = mlflow.set_experiment("/clasificacionisabella")

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name="modelo_clasificacion_nn"):

    # ----- Hiperparámetros -----
    params_nn = {
        "hidden_layer_sizes": (64, 32, 16, 8),
        "activation": "relu",
        "solver": "adam",
        "learning_rate_init": 0.001,
        "max_iter": 500,
        "random_state": 42,
    }
    mlflow.log_params(params_nn)

    # ----- Modelo (CLASIFICACIÓN) ----------------------------
    nn = MLPClassifier(**params_nn)

    #  ----- entrenar usando X_train_proc y y_train (0/1) -----
    nn.fit(X_train_proc, y_train)

    #  ----- predicciones -----
    # probabilidades de clase 1
    y_proba = nn.predict_proba(X_test_proc)[:, 1]
    # predicción binaria (0/1)
    y_pred = nn.predict(X_test_proc)

    # ----- Guardar número de iteraciones ---------------------
    with open("n_iter_clasificacion.txt", "w") as f:
        f.write(str(nn.n_iter_))
    mlflow.log_artifact("n_iter_clasificacion.txt")

    # ----- Guardar loss curve -----
    plt.figure()
    plt.plot(nn.loss_curve_)
    plt.xlabel("Iteraciones")
    plt.ylabel("Loss")
    plt.title("Curva de pérdida (clasificación)")
    plt.savefig("loss_curve_clasificacion.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("loss_curve_clasificacion.png")

    # ----- Matriz de confusión como artifact -----
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Real 0", "Real 1"],
        columns=["Pred 0", "Pred 1"],
    )
    cm_df.to_csv("confusion_matrix.csv")
    mlflow.log_artifact("confusion_matrix.csv")

    # (Opcional) Curva ROC
    from sklearn.metrics import RocCurveDisplay
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title("Curva ROC")
    plt.savefig("roc_curve.png", bbox_inches="tight")
    plt.close()
    mlflow.log_artifact("roc_curve.png")

    # Features como artifact
    with open("features_clasificacion.txt", "w") as f:
        for col in X_train.columns:
            f.write(f"{col}\n")
    mlflow.log_artifact("features_clasificacion.txt")

    # ----- Métricas de clasificación -----
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)

    metrics = {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
    }

    for name, value in metrics.items():
        mlflow.log_metric(name, value)
        print(f"{name.upper()}: {value}")

    # ----- Guardar modelo y preprocesador --------------------
    mlflow.sklearn.log_model(
        sk_model=nn,
        artifact_path="modelo_clasificacion_nn"
    )

    #joblib.dump(preprocesamiento, "preprocesamiento_modelo_clasificacion.pkl")
    #mlflow.log_artifact("preprocesamiento_modelo_clasificacion.pkl")
